In [ ]:
pip install agno dotenv openai pandas openpyxl requests chromadb tmdbv3api pypdf sqlalchemy 


In [1]:
from agno.agent import Agent
from agno.models.message import Message
from agno.run import RunContext
from agno.memory import MemoryManager
from agno.tools.serper import SerperTools
from agno.knowledge.knowledge import Knowledge
from agno.knowledge.embedder.azure_openai import AzureOpenAIEmbedder
from agno.models.azure import AzureOpenAI
from agno.team.team import Team
from agno.models.azure import AzureOpenAI
from agno.team.mode import TeamMode
from agno.vectordb.chroma import ChromaDb
from agno.knowledge.reader.pdf_reader import PDFReader
from agno.db.sqlite import SqliteDb
from tmdbv3api import TMDb, Movie

import os
import pandas as pd
from dotenv import load_dotenv  
from os import getenv
from pathlib import Path  



In [2]:
load_dotenv()
#os.environ["ANTHROPIC_API_KEY"] = getenv("ANTHROPIC_API_KEY")
os.environ["AZURE_OPENAI_API_KEY"] = getenv("AZURE_OPENAI_API_KEY")
os.environ["AZURE_ENDPOINT"] = getenv("AZURE_INFERENCE_ENDPOINT")  
os.environ["AZURE_OPENAI_DEPLOYMENT"] = getenv("LLM_MODEL")

In [3]:
#model = Claude(id="claude-sonnet-4-6")
model = AzureOpenAI(id=getenv("LLM-MODEL"), 
                    api_version=getenv("LLM_MODEL_VERSION"),
                    azure_endpoint=getenv("AZURE_INFERENCE_ENDPOINT"))
response = model.response(
    messages=[
        Message(role="user", content="Hello")
    ]
)
print(response.content)

Hello! How can I assist you today?


In [4]:
# ─── USER CONFIGURATION ──────────────────────────────────────────────────────
DATA_FOLDER   = Path('../data/input')   

USE_CASE_FILE = DATA_FOLDER / 'use_case_Multi_Agent_Movie_RecommendationV2.xlsx'
ABT_FILE      = DATA_FOLDER / 'movie_abt_enriched_FINAL.xlsx'
TAXONOMY_FILE = DATA_FOLDER / 'Movie_Recommendation_TaxonomyV2.xlsx'
OSCARS_PDF    = DATA_FOLDER / 'The_98th_Academy_Awards_2026.pdf'

# SQLite file for Agno session memory
MEMORY_DB     = DATA_FOLDER / 'movie_agent_memory.db'
agent_db = SqliteDb(db_file=MEMORY_DB)


AZURE_OPENAI_API_KEY=os.getenv('AZURE_OPENAI_API_KEY')
AZURE_OPENAI_ENDPOINT=os.getenv('AZURE_OPENAI_ENDPOINT')
AZURE_OPENAI_API_VERSION=os.getenv("LLM_MODEL_VERSION") 
TMDB_API_KEY   = os.getenv('TMDB_API_KEY')
SERPER_API_KEY = os.getenv('SERPER_API_KEY')

LLM_MODEL        = os.getenv("LLM_MODEL")
#EMBED_MODEL      = 'text-embedding-3-small'
MAX_ABT_RESULTS  = 5
MAX_TAG_RESULTS  = 10
RAG_TOP_K        = 4
SESSION_ID       = 'movie-bot-session-001'  # change per user

print(f'Data folder: {DATA_FOLDER.resolve()}')
for f in [USE_CASE_FILE, ABT_FILE, TAXONOMY_FILE, OSCARS_PDF]:
    status = '✅' if f.exists() else '❌ NOT FOUND'
    print(f'  {status}  {f.name}')

Data folder: /Users/nsathi/Library/CloudStorage/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/Agno-Class-exercises/data/input
  ✅  use_case_Multi_Agent_Movie_RecommendationV2.xlsx
  ✅  movie_abt_enriched_FINAL.xlsx
  ✅  Movie_Recommendation_TaxonomyV2.xlsx
  ✅  The_98th_Academy_Awards_2026.pdf


In [5]:
df_instructions = pd.read_excel(USE_CASE_FILE, sheet_name='Agent Instructions')
ORCHESTRATOR_AGENT_PROMPT = df_instructions[df_instructions['Prompt Type'] == 'orchestrator_agent_prompt'].values[0][1]
RECOMMENDER_AGENT_PROMPT = df_instructions[df_instructions['Prompt Type'] == 'recommender_agent_prompt'].values[0][1]
AWARD_AGENT_PROMPT = df_instructions[df_instructions['Prompt Type'] == 'award_agent_prompt'].values[0][1]
WEB_AGENT_PROMPT = df_instructions[df_instructions['Prompt Type'] == 'web_agent_prompt'].values[0][1]
print('Orchestrator agent prompt:')
print(ORCHESTRATOR_AGENT_PROMPT)
print("..............................")
print("Recommender agent prompt: ")
print(RECOMMENDER_AGENT_PROMPT)
print("..............................")
print("Award agent prompt: ")
print(AWARD_AGENT_PROMPT)
print("..............................")
print("Web agent prompt: ")
print(WEB_AGENT_PROMPT)


Orchestrator agent prompt:
You are a friendly and knowledgeable movie expert. Your ONLY role is to help users discover and learn about movies. Do not answer questions unrelated to movies.  Before recommending any movie, collect the user's needs through a brief, engaging conversation. Ask ONE leading question at a time across these dimensions (do not ask all at once):
  • Viewing context: Are you watching alone, with family, friends, or a date?
  • Audience: Who is the audience — adults, teens, young children, mixed group?
  • Genre mood: What genre or emotional mood are you in? (e.g., action, comedy, thriller, romance, drama, sci-fi, documentary)
  • Decade/era: Any preference for era or decade? (classic, 1980s–90s, modern, recent)
  • Country/language: Any preference for country of origin or spoken language?
  • Oscar/award interest: Are you interested in critically acclaimed or award-nominated films?
Stop asking when you have enough information about users' needs and preferences to m

In [10]:

# Your existing knowledge base

award_db = ChromaDb(collection="movie_news", path="tmp/chromadb", persistent_client=True)
award_knowledge = Knowledge(vector_db=award_db)

abt_db = ChromaDb(collection="excel_data", path="tmp/chromadb", persistent_client=True)
abt_knowledge = Knowledge(vector_db=abt_db)

session_db = SqliteDb(
    db_file="tmp/movie_team.db",
    session_table="movie_team_sessions",
)


In [11]:
# 1️⃣ Recommender Agent - Genre understanding, TAG + ABT
recommender_agent = Agent(
    name="Recommender Agent",
    role="Handles movie genre understanding, recommendations using TAG and ABT frameworks",
    model=model,
    knowledge=abt_knowledge,
    db = session_db,
    session_id="March 5-2 2026",
    add_history_to_context=True,
    instructions=[RECOMMENDER_AGENT_PROMPT],
    markdown=True,
)

In [12]:
response = recommender_agent.run(input="Can you recommend  Sci - Fiction movies for me to watch this weekend?", stream=False)
print(response.content)

INFO Found 10 documents

INFO Found 10 documents

Here are some highly rated Sci-Fi movies for you to enjoy:

1. 🎬 Title: Twelve Monkeys (1995)  
   📖 Overview: In the year 2035, convict James Cole reluctantly volunteers to be sent back in time to discover the origin of a deadly virus that wiped out nearly all of the earth’s population and forced the survivors into underground communities. But when Cole is mistakenly sent to 1990 instead of 1996, he’s arrested and locked up in a mental hospital. There he meets psychiatrist Dr. Kathryn Railly and the son of a famous virus expert who may hold the key to the Army of the 12 Monkeys; thought to be responsible for unleashing the killer disease.  
   🏷️ Tagline: The future is history.  
   🎭 Genre: Mystery, Sci-Fi, Thriller  
   ⭐ Rating: 7.6/10 from 8,931 votes  
   📈 Popularity: 9.6682  
   ⏱️ Runtime: 129 minutes  
   🌍 Language: English (US)  
   🖼️ Poster: https://image.tmdb.org/t/p/w500/mKIkGoyuR71qz6FdiEiOjxvBQcS.jpg  

2. 🎬 Title: The City of Lost Children (1995)  
   📖 Overview: A s

In [13]:
chat_history = recommender_agent.get_chat_history()

for m in chat_history:
    print(m)

id='2f8b4f5b-473d-4c65-8aca-8c1ec9b7aa1c' role='user' content=' I would like to watch a newly released movie in theater this weekend on March 7 2026' compressed_content=None name=None tool_call_id=None tool_calls=None audio=None images=None videos=None files=None audio_output=None image_output=None video_output=None file_output=None redacted_reasoning_content=None provider_data=None citations=None reasoning_content=None tool_name=None tool_args=None tool_call_error=None stop_after_tool_call=False add_to_agent_memory=True from_history=False metrics=MessageMetrics(input_tokens=0, output_tokens=0, total_tokens=0, audio_input_tokens=0, audio_output_tokens=0, audio_total_tokens=0, cache_read_tokens=0, cache_write_tokens=0, reasoning_tokens=0, cost=None, timer=None, duration=None, time_to_first_token=None, provider_metrics=None) references=None created_at=1772780714 temporary=False
id='a090875f-5a24-4337-8f53-0f4459bc07a5' role='assistant' content=None compressed_content=None name=None tool_

In [14]:
# 2️⃣ Awards & Knowledge Agent - Oscars / RAG / document Q&A
awards_agent = Agent(
    name="Awards & Knowledge Agent",
    role="Handles Oscars questions, awards history, and document-based Q&A using RAG",
    model=model,
    knowledge=award_knowledge,
    db = session_db,
    session_id="March 5-2 2026",
    add_history_to_context=True,
    search_knowledge=True,
    instructions=[AWARD_AGENT_PROMPT],
    markdown=True,
)

In [15]:
response = awards_agent.run(input="Which movies got nominated for the Oscars this year?", stream=False)
print(response.content)

INFO Found 10 documents

The 98th Academy Awards (Oscars) for 2026, honoring movies released in 2025, have announced the nominees. Here are some highlights:

### Actor in a Leading Role Nominees:
- Timothée Chalamet for "Marty Supreme"
- Leonardo DiCaprio for "One Battle after Another"
- Ethan Hawke for "Blue Moon"
- Michael B. Jordan for "Sinners"
- Wagner Moura for "The Secret Agent"

### Actress in a Leading Role Nominees:
- Jessie Buckley for "Hamnet"
- Rose Byrne

### Actor in a Supporting Role Nominees:
- Benicio Del Toro for "One Battle after Another"
- Jacob Elordi for "Frankenstein"
- Delroy Lindo for "Sinners"
- Sean Penn for "One Battle after Another"
- Stellan Skarsgård for "Sentimental Value"

### Actress in a Supporting Role Nominees:
- Elle Fanning for "Sentimental Value"
- Inga Ibsdotter Lilleaas for "Sentimental Value"
- Amy Madigan for "Weapons"
- Wunmi Mosaku for "Sinners"
- Teyana Taylor for "One Battle after Another"

### Other Notable Nominees:
- Animated Feature Film: "Arco," "Elio," "Kp

In [16]:
# 3️⃣ Web & Streaming Agent - Current releases, streaming availability
web_agent = Agent(
    name="Web & Streaming Agent",
    role="Handles current movie releases, streaming availability, and real-time web searches",
    model=model,
    db = session_db,
    tools=[SerperTools()],
    session_id="March 5-2 2026",
    add_history_to_context=True,
    instructions=[WEB_AGENT_PROMPT],
    markdown=True,
)

In [18]:
response = web_agent.run(input="Can you find information on Peaky Blinders: The Immortal Man", stream=False)
print(response.content)

Since you are interested in sci-fi movies for this weekend, I recommend "Peaky Blinders: The Immortal Man," a 2026 release that blends thrilling drama with historical elements and some supernatural undertones. Here are the details:

🎬 Title: Peaky Blinders: The Immortal Man (2026)  
📖 Overview: Amid the chaos of World War II, Tommy Shelby returns from a self-imposed exile to face his most destructive reckoning yet. With the future of his family and country at stake, Shelby must face his demons and choose whether to confront his legacy or disappear forever.  
🏷️ Tagline: (Not specified)  
🎭 Genre: Drama, Historical, Thriller  
⭐ Rating: Not specified (early release year)  
📈 Popularity: High anticipation as a continuation of the Peaky Blinders saga  
⏱️ Runtime: 1h 52m  
🌍 Language: English  
🖼️ Poster: ![Poster](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRwsY6ULubAFzDNFk9DG7PR_YiQNjcsbJo&s=0)  
🏠 Homepage: Available on Netflix with theatrical release on March 6, 2026, and Ne

In [19]:
# 4️⃣ Orchestrator Team - Routes user queries to specialist agents



movie_team = Team(
    name="Movie Chatbot",
    mode=TeamMode.route,
    model=model,
    members=[recommender_agent, awards_agent, web_agent],
    instructions=[ORCHESTRATOR_AGENT_PROMPT],
    session_id="March 5-2 2026",
    db = session_db,
    show_members_responses=True,
    add_history_to_context=True,
    debug_mode=False,
 #   debug_level=2,
    markdown=True,
)

In [20]:
# Run
response = movie_team.run(" I would like to watch a newly released movie in theater", stream=False)
print(response.content)


For someone interested in watching new releases in theaters this weekend, here are some notable options freshly available in March 2026:

1. Peaky Blinders: The Immortal Man  
   - A dramatic continuation of the Peaky Blinders saga, set during World War II, with Tommy Shelby returning to face new threats.  
   - Release Date: March 6, 2026  

2. Hoppers  
   - A Disney/Pixar animated film, one of the most anticipated new movies this March.  

3. Reminders of Him  
   - A drama starring Maika Monroe, released March 13, 2026.  

4. Undertone  
   - A horror film, also new this March.  

5. Ready or Not 2: Here I Come  
   - A continuation of the popular thriller series, releasing this month.  

6. Project Hail Mary  
   - A sci-fi adventure movie, highly anticipated for March 2026.  

If you want, I can provide detailed information on any of these movies including their synopsis, cast, availability, and where to watch in theaters near you. Would you like a recommendation based on a speci

In [21]:
response = movie_team.run("Where can I watch it in San Diego?", stream=False)
print(response.content)

Since you want to watch "Peaky Blinders: The Immortal Man" in theaters in San Diego, here are some options with showtimes available:

- THE LOT Liberty Station  
  Showtimes available from March 6 to March 12, 2026  
  (You can check their website for exact daily times)  
  Link: https://www.thelotent.com/movie/LibertyStation/Peaky-Blinders-The-Immortal-Man

- UltraStar Cinemas Mission Valley  
  Example Showtime: March 12, 2026 at 7:30 PM in Auditorium 1  
  Link: https://ultrastarmovies.com/ultrastar-movies/tickets/109915

- Silverspot Cinema  
  Also showing the movie with showtimes available (check their site for details)  
  Link: https://silverspot.net/1264604

You can also find tickets and showtimes for other theaters in San Diego on platforms like Fandango and Atom Tickets. Would you like me to help you book tickets or find more detailed showtimes for a specific day?


In [22]:
chat_memory = movie_team.get_chat_history()
print("\nSession Memory:")
for m in chat_memory:
    print("-", m.content)


Session Memory:
-  I would like to watch a newly released movie in theater this weekend on March 7 2026
- None
-  I would like to watch a newly released movie in theater
- None
- Where can I watch it in San Diego?
- None


In [24]:
# This code will demonstrate a real flipped interaction

while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    response = movie_team.run(user_input, stream=False)
    
    print("\nAgent:")
    print(response.content)


    # ✅ Retrieve latest stored memory
    chat_memory = movie_team.get_chat_history()


    print("\nSession Memory:")
    for m in chat_memory:
        print("-", m.content)


    print("\n" + "-"*50 + "\n")
#Initial query "Can you recommend a movie for a date night.  
# I love romantic movies and my spouse likes historical fiction"


Your input:  Can you recommend a movie for a date night.   # I love romantic movies and my spouse likes historical fiction

Agent:
I understand you both enjoy romantic movies and historical fiction. Do you have a preference for the era or decade of the historical setting for your date night movie? For example, are you interested in something set in the Regency era, Victorian times, World War periods, or another time?

Session Memory:
-  I would like to watch a newly released movie in theater this weekend on March 7 2026
- None
-  I would like to watch a newly released movie in theater
- None
- Where can I watch it in San Diego?
- None
- Can you recommend a movie for a date night.   # I love romantic movies and my spouse likes historical fiction
- That sounds like a lovely date night! You enjoy romantic movies, and your spouse prefers historical fiction. To find the perfect movie blend for you both, do you have any preference for the era or decade of the historical setting? For example,

INFO Found 10 documents


Agent:
For a romantic date night with a historical fiction setting in the Victorian era, here are some lovely movie options:

1. 🎬 Sense and Sensibility (1995)  
   📖 Overview: The Dashwood sisters, sensible Elinor and passionate Marianne, learn that their prospects of marriage seem doomed by their family’s sudden loss of fortune. As they struggle to find romantic fulfillment in a society obsessed with financial and social status, they must learn to mix sense with sensibility in their dealings with both money and men.  
   🏷️ Tagline: Lose your heart and come to your senses.  
   🎭 Genre: Drama, Romance  
   ⭐ Rating: 7.4/10 from 1,804 votes  
   ⏱️ Runtime: 136 minutes  
   🌍 Language: English (GB, US)  
   🖼️ Poster: https://image.tmdb.org/t/p/w500/cFnyfWHL6HjOOVal6lp8e5gqibK.jpg  

2. 🎬 Persuasion (1995)  
   📖 Overview: In early 19th-century England, dire financial straits reacquaint the aristocratic Anne Elliot with her wealthy ex-fiance Frederick Wentworth. The two must choose b